# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

/var/folders/9p/f66yhymx35939zzlg51g0rrc0000gn/T/ipykernel_18377/3307975217.py:5: DeprecationWarning: Please import from 'ax.generation_strategy.generation_strategy' instead of 'ax.modelbridge.generation_strategy'. The latter is deprecated and will be removed in a future release.
  from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


# generate recommendations

In [2]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 9
Very Important: Please Confirm the Iteration Number is Iteration 9
Very Important: Please Confirm the Iteration Number is Iteration 9


In [3]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = "IBP", bopt = 1, n_trials=3)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

**************************************************************************************************************

Generating Bayesian Optimization trialsfor
Drug name:  Ibuprofen IBP  | Iteration:  9

**************************************************************************************************************


[INFO 07-02 09:41:07] ax.service.ax_client: Generated new trial 27 with parameters {'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 100, 's2': 100, 's3': 100, 's4': 0, 's5': 0, 's6': 0, 's7': 0, 's8': 0, 'surfactant_conc': 91, 'drug_conc': 100} using model SAASBO.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/core/data.py:293: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
[INFO 07-02 09:45:05] ax.service.ax_client: Generated new trial 28 with parameters {'Drug_MW': 0.2

Time taken for optimization: 11.48 mins
Time taken for optimization: 688.8000000000001 seconds


# process results

In [4]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 47, 's2': 60, 's3': 50, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 'surfactant_conc': 85, 'drug_conc': 100})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 74, 's2': 0, 's3': 88, 's4': 66, 's5': 22, 's6': 52, 's7': 58, 's8': 79, 'surfactant_conc': 35, 'drug_conc': 100})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 78, 's2': 88, 's3': 1, 's4': 3, 's5': 32, 's6': 94, 's7': 44, 's8': 58, 'surfactant_conc': 1, 'drug_conc': 100})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', paramete

In [5]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [6]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: D1
Deep plate will start at: F7

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [7]:
hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_9.py


In [ ]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

In [ ]:
results = hf.build_results(n, df_conc, df_absorbance)
results

In [ ]:
norm_results = hf.normalize_data(results, 'normalize')

In [ ]:
norm_results

# load the results to the optimizer

In [ ]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client